In [12]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, precision_recall_curve, f1_score, roc_auc_score, average_precision_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device}")

Execution Device: cpu


In [13]:
df = pd.read_csv("data/secom_telemetry_modified.csv")
print("Raw DataFrame shape:", df.shape)

# 1. Drop timestamp/datetime columns if present
non_numeric_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print("Non-numeric/string columns detected:", non_numeric_cols)

# Drop timestamp columns that aren't the target
for col in non_numeric_cols:
    if col.lower() in ['time', 'date', 'timestamp'] or df[col].astype(str).str.contains(r'[-/:]').any():
        print(f"Dropping timestamp column: {col}")
        df = df.drop(columns=[col])

# 2. Identify the target label column
target_candidates = [c for c in ['Defect', 'target', 'Label', 'Pass/Fail', 'class'] if c in df.columns]
if target_candidates:
    target_col = target_candidates[0]
else:
    # SECOM standard puts the target label as the last column
    target_col = df.columns[-1]

print(f"Target column identified as: '{target_col}'")

# Ensure target values are numeric 0 and 1
df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
unique_targets = set(df[target_col].dropna().unique())
print(f"Unique target values before mapping: {unique_targets}")

# SECOM labels are typically -1 (Pass/Normal) and 1 (Fail/Defect)
if -1 in unique_targets or 1 in unique_targets:
    df[target_col] = df[target_col].map({-1: 0, 1: 1}).fillna(df[target_col])

# Drop any rows where target is missing
df = df.dropna(subset=[target_col]).reset_index(drop=True)

# 3. STRICT CAUSALITY: Chronological Split (80% Train, 20% Test)
split_idx = int(0.8 * len(df))
train_df = df.iloc[:split_idx].copy().reset_index(drop=True)
test_df = df.iloc[split_idx:].copy().reset_index(drop=True)

# 4. BUSINESS LOGIC: Legacy Probe Station PS-L handling
def process_probe_station(data):
    df_out = data.copy()
    
    # 1. Flag wafers passing through legacy PS-L (sensor_0 == 0.0 is a sentinel)
    is_ps_l = (df_out["sensor_0"] == 0.0).astype(float)
    df_out["is_ps_l"] = is_ps_l
    
    # 2. Extract categorical error codes (100.0, 200.0, 300.0) without continuous pollution
    for code in [100.0, 200.0, 300.0]:
        df_out[f"ps_l_err_{int(code)}"] = ((is_ps_l == 1.0) & (df_out["sensor_1"] == code)).astype(float)
        
    # 3. For legacy wafers, replace sentinel sensor_0 and categorical sensor_1 with NaN
    # so they are not included in continuous feature scaling
    df_out.loc[df_out["is_ps_l"] == 1.0, "sensor_0"] = np.nan
    df_out.loc[df_out["is_ps_l"] == 1.0, "sensor_1"] = np.nan
    return df_out

train_proc = process_probe_station(train_df)
test_proc = process_probe_station(test_df)

feature_cols = [c for c in train_proc.columns if c != target_col]
continuous_cols = [c for c in feature_cols if not (c.startswith("ps_l_") or c == "is_ps_l")]
cat_cols = [c for c in feature_cols if c not in continuous_cols]

# 5. Fit only on Train
train_medians = train_proc[continuous_cols].median()
train_means = train_proc[continuous_cols].mean()
train_stds = train_proc[continuous_cols].std().replace(0, 1.0).fillna(1.0)

for col in continuous_cols:
    med = train_medians[col] if pd.notnull(train_medians[col]) else 0.0
    mu = train_means[col] if pd.notnull(train_means[col]) else 0.0
    sigma = train_stds[col]
    
    train_proc[col] = (train_proc[col].fillna(med) - mu) / sigma
    test_proc[col] = (test_proc[col].fillna(med) - mu) / sigma

# Fill categorical indicators if missing
train_proc[cat_cols] = train_proc[cat_cols].fillna(0.0)
test_proc[cat_cols] = test_proc[cat_cols].fillna(0.0)

# 6. Tensor conversion with explicit numeric casting
X_train_np = train_proc[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0).values.astype(np.float32)
y_train_np = train_proc[target_col].values.astype(np.float32)

X_test_np = test_proc[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0).values.astype(np.float32)
y_test_np = test_proc[target_col].values.astype(np.float32)

X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32).view(-1, 1)

print("="*50)
print(f"X_train Shape: {X_train.shape}, y_train Shape: {y_train.shape}")
print(f"X_test Shape:  {X_test.shape},  y_test Shape:  {y_test.shape}")
print(f"Natural Test Defect Prevalence: {y_test.mean().item() * 100:.2f}%")

Raw DataFrame shape: (1567, 592)
Non-numeric/string columns detected: ['timestamp']
Dropping timestamp column: timestamp
Target column identified as: 'label'
Unique target values before mapping: {np.int64(1), np.int64(-1)}
X_train Shape: torch.Size([1253, 594]), y_train Shape: torch.Size([1253, 1])
X_test Shape:  torch.Size([314, 594]),  y_test Shape:  torch.Size([314, 1])
Natural Test Defect Prevalence: 5.41%


In [14]:
class WaferSupervisedAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        # Encoder Network
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(128, latent_dim),
            nn.LeakyReLU(0.2)
        )
        # Decoder Network (Reconstruction)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, input_dim)
        )
        # Classification Head (Supervised Anomaly Prediction)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 1)  # Outputs raw logit
        )

    def forward(self, x):
        z = self.encoder(x)
        x_rec = self.decoder(z)
        logits = self.classifier(z)
        return x_rec, logits

In [15]:
# Compute positive class weighting to handle imbalance during training only
n_pos = (y_train == 1).sum().item()
n_neg = (y_train == 0).sum().item()
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=device)

model = WaferSupervisedAE(input_dim=X_train.shape[1], latent_dim=32).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

recon_loss_fn = nn.MSELoss()
class_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

epochs = 30
model.train()
for epoch in range(1, epochs + 1):
    total_loss, total_recon, total_class = 0.0, 0.0, 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        x_rec, logits = model(batch_x)
        
        # STRICT TENSOR ASSERTIONS: Guard against (N,) vs (N, 1) silent broadcasting
        assert logits.shape == batch_y.shape, f"Logit shape {logits.shape} != Target shape {batch_y.shape}"
        assert x_rec.shape == batch_x.shape, f"Recon shape {x_rec.shape} != Input shape {batch_x.shape}"
        
        l_recon = recon_loss_fn(x_rec, batch_x)
        l_class = class_loss_fn(logits, batch_y)
        loss = l_recon + 2.0 * l_class
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += l_recon.item()
        total_class += l_class.item()
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs:02d}] - Loss: {total_loss/len(train_loader):.4f} (Recon: {total_recon/len(train_loader):.4f}, Class: {total_class/len(train_loader):.4f})")

Epoch [01/30] - Loss: 3.3433 (Recon: 0.7525, Class: 1.2954)
Epoch [10/30] - Loss: 1.0388 (Recon: 0.6085, Class: 0.2151)
Epoch [20/30] - Loss: 0.7038 (Recon: 0.5587, Class: 0.0726)
Epoch [30/30] - Loss: 0.5698 (Recon: 0.5281, Class: 0.0208)


In [16]:
model.eval()
with torch.no_grad():
    x_test_dev = X_test.to(device)
    _, test_logits = model(x_test_dev)
    probs = torch.sigmoid(test_logits).cpu().numpy().flatten()
    y_true = y_test.numpy().flatten()

prec, rec, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

y_pred = (probs >= best_threshold).astype(int)

measured_f1 = f1_score(y_true, y_pred)
pr_auc = average_precision_score(y_true, probs)
roc_auc = roc_auc_score(y_true, probs)

print("="*60)
print(f"EVALUATION AT NATURAL PREVALENCE ({y_true.mean()*100:.2f}%)")
print("="*60)
print(f"Optimal Decision Threshold : {best_threshold:.4f}")
print(f"Defect Class F1 Score      : {measured_f1:.4f}")
print(f"PR-AUC (Average Precision) : {pr_auc:.4f}")
print(f"ROC-AUC Score              : {roc_auc:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Defective (1)"]))

EVALUATION AT NATURAL PREVALENCE (5.41%)
Optimal Decision Threshold : 0.7823
Defect Class F1 Score      : 0.3077
PR-AUC (Average Precision) : 0.2354
ROC-AUC Score              : 0.7851

Classification Report:

               precision    recall  f1-score   support

   Normal (0)       0.96      0.98      0.97       297
Defective (1)       0.44      0.24      0.31        17

     accuracy                           0.94       314
    macro avg       0.70      0.61      0.64       314
 weighted avg       0.93      0.94      0.93       314

